### Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

### Autoencoders

In [ ]:
class Autoencoder(nn.Module):
    def __init__(
        self,
        dim: tuple
    ) -> None:
        super().__init__()

        input_dim, bottleneck_dim = dim

        self.encoder = nn.Linear(input_dim, bottleneck_dim)
        self.sigmoid = nn.Sigmoid()
        self.decoder = nn.Linear(bottleneck_dim, input_dim)

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:
        if x.ndim == 1:
            x = x.unsqueeze(0)
        z = self.encoder(x)
        a = self.sigmoid(z)
        z = self.decoder(a)
        return z, a.mean()

    # def get_activations(
    #     self,
    #     x: torch.Tensor
    # ) -> torch.Tensor:
    #     with torch.no_grad():
    #         if x.ndim == 1:
    #             x = x.unsqueeze(0)
    #         z = self.encoder(x)
    #         a = self.sigmoid(z)

    #         return a

In [ ]:
class SparseMSELoss(nn.Module):
    def __init__(
        self,
        beta: float = 1.0
    ) -> None:
        super().__init__()
        self.mse_loss = nn.MSELoss()
        self.beta = beta

    def forward(
        self,
        x_recon: torch.Tensor,
        x_original: torch.Tensor,
        p: torch.Tensor,
        p_hat: torch.Tensor,
    ) -> torch.Tensor:
        mse_loss = self.mse_loss(x_recon, x_original)

        p     = torch.clamp(p,     min=1e-7, max=1 - 1e-7)
        p_hat = torch.clamp(p_hat, min=1e-7, max=1 - 1e-7)

        kl_divergence = (
            p       * (torch.log(p)     - torch.log(p_hat)) +
            (1 - p) * (torch.log(1 - p) - torch.log(1 - p_hat))
        ).mean()

        total_loss = mse_loss + self.beta * kl_divergence
        return total_loss

In [ ]:
normal_autoencoder = Autoencoder(dim=(100, 80)).to(device)
normal_criterion   = nn.MSELoss()
normal_optimizer   = torch.optim.Adam(normal_autoencoder.parameters(), lr=1e-3)

sparse_autoencoder = Autoencoder(dim=(100, 80)).to(device)
sparse_criterion   = SparseMSELoss(beta=0.00001)
sparse_optimizer   = torch.optim.Adam(normal_autoencoder.parameters(), lr=1e-3)

In [ ]:
batch_size = 128

mnist_mean = 0.1307
mnist_std = 0.3081
mnist_transform = transforms.Compose([
    transforms.Resize((10, 10)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mnist_mean], std=[mnist_std]),
    transforms.Lambda(lambda x: x.view(-1)),
])

mnist_train = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=mnist_transform,
)

train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)

### Training

In [ ]:
num_epochs = 5
normal_losses = []
sparse_losses = []

normal_autoencoder.train()
sparse_autoencoder.train()
epochbar = tqdm(range(1, num_epochs + 1), desc="Epochs")

p = torch.tensor(0.05)

for epoch in epochbar:
    normal_epoch_loss = 0.0
    sparse_epoch_loss = 0.0
    for batch_inputs, _ in train_loader:
        batch_input = batch_inputs.to(device)
        normal_optimizer.zero_grad()
        sparse_optimizer.zero_grad()

        normal_x_recon, _     = normal_autoencoder(batch_input)
        sparse_x_recon, p_hat = sparse_autoencoder(batch_input)

        normal_loss = normal_criterion(normal_x_recon, batch_input)
        sparse_loss = sparse_criterion(sparse_x_recon, batch_input, p, p_hat)

        normal_loss.backward()
        sparse_loss.backward()

        normal_optimizer.step()
        sparse_optimizer.step()

        normal_batch_loss = normal_loss.item()
        sparse_batch_loss = sparse_loss.item()

        normal_epoch_loss += normal_batch_loss * batch_inputs.size(0)
        sparse_epoch_loss += sparse_batch_loss * batch_inputs.size(0)

        epochbar.set_postfix({
            "normal_loss": f"{normal_batch_loss:.4f}",
            "sparse_loss": f"{sparse_batch_loss:.4f}"
        })

    normal_batch_loss /= len(train_loader.dataset)
    sparse_batch_loss /= len(train_loader.dataset)

    normal_losses.append(normal_batch_loss)
    sparse_losses.append(sparse_batch_loss)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(range(1, num_epochs + 1), normal_losses, "o-")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean Train Loss")
axes[0].set_title("Normal Autoencoder Training Performance")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, num_epochs + 1), sparse_losses, "o-", color='orange')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Train Loss")
axes[1].set_title("Sparse Autoencoder Training Performance")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Visualization

In [ ]:
# TODO: Add visualization code